# Tutorial: use all six DAP breed-prediction modes

This standalone tutorial shows how to:

1. install the repository and notebook dependencies;
2. inspect the bundled chromosome-wise Parquet files safely;
3. inspect the example CSV and label formats;
4. configure and launch each of Modes 1 through 6 through either the Python
   API or command-line interface (CLI); and
5. inspect the files produced by a run.

All paths are resolved from the repository root. **No model-training mode runs by
default.** Each mode cell prints its generated configuration and exact command.
To run a mode, add its number to `RUN_MODES` in the control cell and then execute
that mode's cell.

Modes 1, 2, 3, 5, and 6 train random-forest models. Mode 5 reads all 54,143 SNPs
and has the largest memory requirement. Start with the small Mode 4 demonstration.

## 1. Locate the repository

Start Jupyter from either the repository root or `notebooks/`. The next cell
locates the repository without requiring package imports.

In [1]:
from pathlib import Path

starts = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (path for path in starts if (path / "pyproject.toml").is_file() and (path / "main.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Repository root not found. Clone DAP_breed_prediction and start Jupyter "
        "from the repository root or its notebooks directory."
    )

print(f"Repository detected: {REPO_ROOT.name}")

Repository detected: DAP_breed_prediction


## 2. Install the required packages

For a new checkout, the recommended terminal setup is:

```bash
git clone https://github.com/AkeyLab/DAP_breed_prediction.git
cd DAP_breed_prediction
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -e ".[figures]"
python -m jupyter lab notebooks/tutorial_all_modes.ipynb
```

Alternatively, set `INSTALL_PACKAGES = True` below and run the cell once. It
installs the project plus the optional Jupyter/figure dependencies into the
current kernel environment. Restart the kernel after installation if imports
were previously unavailable.

In [2]:
import shlex
import subprocess
import sys

INSTALL_PACKAGES = False
install_command = [sys.executable, "-m", "pip", "install", "-e", f"{REPO_ROOT}[figures]"]

if INSTALL_PACKAGES:
    subprocess.check_call(install_command, cwd=REPO_ROOT)
    print("Installation complete. Restart the kernel before continuing if needed.")
else:
    print('Installation skipped. To install, set INSTALL_PACKAGES = True.')
    print('Terminal equivalent: python -m pip install -e ".[figures]"')

Installation skipped. To install, set INSTALL_PACKAGES = True.
Terminal equivalent: python -m pip install -e ".[figures]"


In [3]:
import platform
import re
import shlex
import subprocess
import sys

import joblib
import matplotlib
import numpy as np
import pandas as pd
import polars as pl
import sklearn
import yaml
from IPython.display import display
from dap_breed_prediction import run_mode

print(f"Python:        {platform.python_version()}")
print(f"pandas:        {pd.__version__}")
print(f"polars:        {pl.__version__}")
print(f"scikit-learn:  {sklearn.__version__}")

Python:        3.9.25
pandas:        2.3.3
polars:        1.36.1
scikit-learn:  1.6.1


## 3. Check the bundled inputs

The repository contains two input groups:

- `data/folder_of_54143_SNPs/`: 38 chromosome-wise standardized DAP training
  matrices used by Modes 1, 2, 3, and 5;
- CSV/model assets under `data/` and `model/`: toy input, labels, breed list,
  fixed PCA matrices, and the archived model used by the examples and Mode 6.

The checks below read metadata only and do not load the complete genotype
matrix into memory.

In [4]:
DATA_DIR = REPO_ROOT / "data"
PARQUET_DIR = DATA_DIR / "folder_of_54143_SNPs"


def chromosome_number(path):
    match = re.search(r"_ch(\d+)_", path.name)
    if match is None:
        raise ValueError(f"Cannot identify chromosome number in {path.name}")
    return int(match.group(1))


parquet_files = sorted(
    PARQUET_DIR.glob("X_SNP_ch*_pruned_v3_std.parquet"),
    key=chromosome_number,
)
required_files = {
    "Toy genotypes": DATA_DIR / "Toy_X_snps.csv",
    "Toy labels": DATA_DIR / "Toy_Y_labels.csv",
    "Toy breed list": DATA_DIR / "Toy_a_short_breed_list.txt",
    "100-class labels": DATA_DIR / "y_combined_100.csv",
    "Mode 6 training PCs": DATA_DIR / "X_train_SNP_WG_prune_v3_1_std_pca_100.csv",
    "Mode 6 test PCs": DATA_DIR / "X_test_SNP_WG_prune_v3_1_std_pca_100.csv",
    "Archived 100-class model": REPO_ROOT / "model" / "regressor_model0_4-PCA100.pkl",
}

if len(parquet_files) != 38:
    raise FileNotFoundError(f"Expected 38 chromosome Parquet files, found {len(parquet_files)}")
missing = [label for label, path in required_files.items() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing bundled inputs: {missing}")

inventory = pd.DataFrame(
    [(label, path.relative_to(REPO_ROOT).as_posix(), path.stat().st_size / 2**20)
     for label, path in required_files.items()],
    columns=["Input", "Repository path", "Size (MiB)"],
)
display(inventory.round({"Size (MiB)": 2}))
print(f"Chromosome Parquet files: {len(parquet_files)}")
print(f"Combined Parquet size: {sum(path.stat().st_size for path in parquet_files) / 2**20:.1f} MiB")

,Input,Repository path,Size (MiB)
0,Toy genotypes,data/Toy_X_snps.csv,0.26
1,Toy labels,data/Toy_Y_labels.csv,0.00
2,Toy breed list,data/Toy_a_short_breed_list.txt,0.00
3,100-class labels,data/y_combined_100.csv,2.55
4,Mode 6 training PCs,data/X_train_SNP_WG_prune_v3_1_std_pca_100.csv,8.36
5,Mode 6 test PCs,data/X_test_SNP_WG_prune_v3_1_std_pca_100.csv,3.61
6,Archived 100-class model,model/regressor_model0_4-PCA100.pkl,31.21


Chromosome Parquet files: 38
Combined Parquet size: 134.9 MiB


## 4. View a Parquet file

Use Polars to inspect only the columns and rows you need. Avoid loading all 38
files and all 54,143 SNP columns merely to preview the data.

In [5]:
chromosome_rows = []
for path in parquet_files:
    schema = pl.read_parquet_schema(path)
    columns = list(schema)
    row_count = pl.scan_parquet(path).select(pl.len()).collect().item()
    chromosome_rows.append({
        "chromosome": chromosome_number(path),
        "file": path.name,
        "dogs": row_count,
        "SNP columns": len(columns) - 1,
        "size (MiB)": path.stat().st_size / 2**20,
    })

chromosome_summary = pd.DataFrame(chromosome_rows)
display(chromosome_summary.head())
print(f"Rows per file: {sorted(chromosome_summary['dogs'].unique())}")
print(f"Total SNP columns: {chromosome_summary['SNP columns'].sum():,}")
print("Use `display(chromosome_summary)` to view all 38 rows.")

,chromosome,file,dogs,SNP columns,size (MiB)
0,1,X_SNP_ch1_pruned_v3_std.parquet,7618,2137,5.296098
1,2,X_SNP_ch2_pruned_v3_std.parquet,7618,1773,4.398164
2,3,X_SNP_ch3_pruned_v3_std.parquet,7618,1785,4.427796
3,4,X_SNP_ch4_pruned_v3_std.parquet,7618,1712,4.247811
4,5,X_SNP_ch5_pruned_v3_std.parquet,7618,2135,5.286780


Rows per file: [np.int64(7618)]
Total SNP columns: 54,143
Use `display(chromosome_summary)` to view all 38 rows.


In [6]:
example_parquet = parquet_files[0]
example_schema = pl.read_parquet_schema(example_parquet)
example_snps = [column for column in example_schema if column != "dog_id"][:5]
preview_columns = ["dog_id", *example_snps]

print(f"File: {example_parquet.relative_to(REPO_ROOT)}")
print(f"Columns in this chromosome: {len(example_schema):,}")
display(pl.read_parquet(example_parquet, columns=preview_columns, n_rows=5))

File: data/folder_of_54143_SNPs/X_SNP_ch1_pruned_v3_std.parquet
Columns in this chromosome: 2,138


dog_id,chr1:5753:G:A,chr1:60406:C:T,chr1:63575:G:C,chr1:71711:A:G,chr1:99638:G:A
str,f64,f64,f64,f64,f64
"""100014""",1.24208,-1.688129,-0.130245,-1.971228,-1.670919
"""100030""",-0.221745,-0.226082,-0.130245,-0.312095,-0.032265
"""100033""",1.24208,1.235965,-0.130245,1.347038,-0.032265
"""100035""",-0.221745,-0.226082,-0.130245,1.347038,-1.670919
"""100044""",1.24208,-1.688129,-0.130245,-0.312095,-0.032265


Additional useful Polars patterns:

```python
# Schema without loading observations
schema = pl.read_parquet_schema(example_parquet)

# Select a few columns lazily
subset = (
    pl.scan_parquet(example_parquet)
    .select(["dog_id", *example_snps])
    .head(10)
    .collect()
)

# Retrieve one dog without materializing every row
one_dog = (
    pl.scan_parquet(example_parquet)
    .filter(pl.col("dog_id") == "100030")
    .select(["dog_id", *example_snps])
    .collect()
)
```

Pandas can also read Parquet with `pd.read_parquet()`, but it requires an
additional Parquet engine such as `pyarrow`. Polars is already a project
dependency.

## 5. Inspect the example CSV inputs

`SNP_csv_path` must point to a CSV containing `dog_id` and SNP columns named
`chr<chromosome>:<position>:<reference>:<alternate>`. `label_path` must contain
`dog_id` and `label`; mixed labels use `Breed A / Breed B`. A breed-list text
file contains one breed per line.

In [7]:
toy_x_path = DATA_DIR / "Toy_X_snps.csv"
toy_y_path = DATA_DIR / "Toy_Y_labels.csv"
breed_list_path = DATA_DIR / "Toy_a_short_breed_list.txt"

toy_header = pd.read_csv(toy_x_path, nrows=0)
toy_preview = pd.read_csv(toy_x_path, usecols=list(toy_header.columns[:6]), nrows=5)
toy_labels = pd.read_csv(toy_y_path)
toy_breeds = [line.strip() for line in breed_list_path.read_text().splitlines() if line.strip()]

display(toy_preview)
display(toy_labels.head())
print(f"Toy samples: {len(toy_labels)}")
print(f"Toy SNP columns: {sum(column.startswith('chr') for column in toy_header.columns)}")
print(f"Breeds in the example breed list: {len(toy_breeds)}")

,dog_id,chr10:28623837:T:A,chr10:42424680:G:A,chr10:51884158:C:G,chr10:37569583:T:A,chr10:6233388:G:A
0,106610,-0.154391,1.115068,1.232223,-1.448990,-0.118378
1,109622,-1.481878,-0.225195,1.232223,1.214075,-0.118378
2,11086,1.173095,-0.225195,-0.081389,1.214075,1.198123
3,110939,-1.481878,-1.565458,1.232223,-0.117457,-0.118378
4,113788,-1.481878,-1.565458,1.232223,-0.117457,-1.434878


,dog_id,label
0,109622,Australian Shepherd
1,110939,Chinese Shar-Pei
2,113788,Siberian Husky
3,114741,German Shepherd Dog
4,116669,German Shorthaired Pointer


Toy samples: 50
Toy SNP columns: 266
Breeds in the example breed list: 41


## 6. Python API, CLI, and safe execution helper

The command-line flags determine the mode. Modes 1–3 all use `-inference`, but
the available configuration keys distinguish them. Despite the flag name,
Modes 1–3 first train a model from the bundled DAP data and then infer the
provided samples.

| Mode | Flags | Purpose | Main inputs |
| --- | --- | --- | --- |
| 1 | `-inference` | Default 14-breed panel | SNP CSV |
| 2 | `-inference` | User-selected breed panel | SNP CSV + breed list |
| 3 | `-inference` | Selected panel with evaluation | SNP CSV + labels, optional breed list |
| 4 | `-train -inference` | Train/test split of a provided dataset | SNP CSV + labels |
| 5 | `-train` | Train from the complete DAP panel | Breed list + all 38 Parquet files |
| 6 | `-reproduce` | Reproduce the paper's 100-class model | Fixed bundled PCA train/test matrices |

Set `RUN_MODES` to the modes you intentionally want to execute. Each mode uses
its own `results/tutorial_mode_<n>/` directory. Existing results may be reused;
choose a new result path for an independent run.

The notebook calls the public Python API:

```python
from dap_breed_prediction import run_mode
run_mode(4, config, base_dir=REPO_ROOT)
```

The CLI delegates to the same API, so both interfaces execute identical
validation and pipeline logic.

In [8]:
# Examples: {4} runs the quick toy demo; {3, 4} runs Modes 3 and 4.
# Keep this empty to preview configurations and commands without training.
RUN_MODES = set()

MODE_FLAGS = {
    1: ["-inference"],
    2: ["-inference"],
    3: ["-inference"],
    4: ["-train", "-inference"],
    5: ["-train"],
    6: ["-reproduce"],
}


def mode_config(mode):
    template = REPO_ROOT / "configs" / f"config_mode_{mode}_template.yml"
    with template.open() as handle:
        config = yaml.safe_load(handle) or {}
    config["result_folder_path"] = f"./results/tutorial_mode_{mode}"
    return config


def show_or_run_mode(mode):
    if mode not in MODE_FLAGS:
        raise ValueError(f"Unknown mode: {mode}")

    config = mode_config(mode)
    template_path = Path("configs") / f"config_mode_{mode}_template.yml"
    display_command = ["python", "main.py", *MODE_FLAGS[mode], "-config_path", template_path.as_posix()]

    print(f"Mode {mode} configuration:\n")
    print(yaml.safe_dump(config, sort_keys=False).rstrip())
    print(f"\nPython API:\nrun_mode({mode}, config, base_dir=REPO_ROOT)")
    print(f"\nCLI using the bundled template:\n{shlex.join(display_command)}")

    if mode not in RUN_MODES:
        print(f"\nSkipped. Add {mode} to RUN_MODES to execute this mode.")
        return None

    return run_mode(mode, config, base_dir=REPO_ROOT)

## 7. Mode 1: default breed-panel inference

Mode 1 selects the default 14 breeds plus `Unknown`, trains from DAP dogs using
SNPs that overlap `Toy_X_snps.csv`, and predicts the supplied samples.

Required configuration: `result_folder_path`, `SNP_csv_path`.

In [9]:
mode_1_result = show_or_run_mode(1)

Mode 1 configuration:

result_folder_path: ./results/tutorial_mode_1
SNP_csv_path: ./data/Toy_X_snps.csv

Python API:
run_mode(1, config, base_dir=REPO_ROOT)

CLI using the bundled template:
python main.py -inference -config_path configs/config_mode_1_template.yml

Skipped. Add 1 to RUN_MODES to execute this mode.


## 8. Mode 2: inference with a breed list

Mode 2 derives the output classes from the supplied breed-list text file,
trains on matching DAP dogs, and predicts the supplied samples.

Required configuration: `result_folder_path`, `SNP_csv_path`,
`breed_list_text_path`.

In [10]:
mode_2_result = show_or_run_mode(2)

Mode 2 configuration:

result_folder_path: ./results/tutorial_mode_2
SNP_csv_path: ./data/Toy_X_snps.csv
breed_list_text_path: ./data/Toy_a_short_breed_list.txt

Python API:
run_mode(2, config, base_dir=REPO_ROOT)

CLI using the bundled template:
python main.py -inference -config_path configs/config_mode_2_template.yml

Skipped. Add 2 to RUN_MODES to execute this mode.


## 9. Mode 3: inference and evaluation

Mode 3 adds known labels, so it reports strict and loose accuracy and writes
evaluation tables/plots. If `breed_list_text_path` is supplied, that list
defines the classes; otherwise classes are inferred from `label_path`.

Required configuration: `result_folder_path`, `SNP_csv_path`, `label_path`.
The breed list is optional.

In [11]:
mode_3_result = show_or_run_mode(3)

Mode 3 configuration:

result_folder_path: ./results/tutorial_mode_3
SNP_csv_path: ./data/Toy_X_snps.csv
breed_list_text_path: ./data/Toy_a_short_breed_list.txt
label_path: ./data/Toy_Y_labels.csv

Python API:
run_mode(3, config, base_dir=REPO_ROOT)

CLI using the bundled template:
python main.py -inference -config_path configs/config_mode_3_template.yml

Skipped. Add 3 to RUN_MODES to execute this mode.


## 10. Mode 4: train/test on a provided dataset

Mode 4 is the recommended first run. It uses only the small toy CSV and labels,
creates a reproducible train/test split, optionally applies PCA, trains a model,
and evaluates held-out samples. It does not use the chromosome Parquet files.

Required configuration: `result_folder_path`, `SNP_csv_path`, `label_path`.
Optional settings include `breed_list_text_path`, `pca_components`,
`random_state`, and `test_size`.

To run it, set `RUN_MODES = {4}`, execute the control/helper cell again, and
then execute the next cell.

In [12]:
mode_4_result = show_or_run_mode(4)

Mode 4 configuration:

result_folder_path: ./results/tutorial_mode_4
SNP_csv_path: ./data/Toy_X_snps.csv
breed_list_text_path: ./data/Toy_a_short_breed_list.txt
label_path: ./data/Toy_Y_labels.csv
pca_components: 0.95
random_state: 42
test_size: 0.3

Python API:
run_mode(4, config, base_dir=REPO_ROOT)

CLI using the bundled template:
python main.py -train -inference -config_path configs/config_mode_4_template.yml

Skipped. Add 4 to RUN_MODES to execute this mode.


## 11. Mode 5: train on the complete DAP panel

Mode 5 loads all 38 Parquet files and can use all 54,143 SNPs. This is the most
memory-intensive workflow and may take substantially longer than the toy demo.
Use a compute node with sufficient RAM rather than a constrained laptop.

Required configuration: `result_folder_path`, `breed_list_text_path`.
Optional settings include `pca_components`, `random_state`, and `test_size`.

In [13]:
mode_5_result = show_or_run_mode(5)

Mode 5 configuration:

result_folder_path: ./results/tutorial_mode_5
breed_list_text_path: ./data/Toy_a_short_breed_list.txt
pca_components: 0.95
random_state: 42
test_size: 0.4

Python API:
run_mode(5, config, base_dir=REPO_ROOT)

CLI using the bundled template:
python main.py -train -config_path configs/config_mode_5_template.yml

Skipped. Add 5 to RUN_MODES to execute this mode.


## 12. Mode 6: reproduce the 100-class paper model

Mode 6 uses the fixed bundled 100-PC training/test matrices and paper settings
(`pca_components=100`, `random_state=42`). It trains a new 100-output random
forest and evaluates the fixed test set. It does not use the chromosome
Parquet files. The reference runtime is approximately 4–5 minutes, depending
on CPU resources.

Required configuration: only `result_folder_path`.

In [14]:
mode_6_result = show_or_run_mode(6)

Mode 6 configuration:

result_folder_path: ./results/tutorial_mode_6

Python API:
run_mode(6, config, base_dir=REPO_ROOT)

CLI using the bundled template:
python main.py -reproduce -config_path configs/config_mode_6_template.yml

Skipped. Add 6 to RUN_MODES to execute this mode.


## 13. Inspect run outputs

Each mode writes under its configured result directory:

- `process.log`: parameters, dimensions, threshold, and metrics;
- `Model/`: prediction model and optional scaler/PCA model;
- `Table/`: raw/transformed predictions and optional evaluation tables;
- `Figure/`: optional SVG output.

Run the next cell after executing a mode. Change `MODE_TO_INSPECT` as needed.

In [15]:
MODE_TO_INSPECT = 4
result_dir = REPO_ROOT / "results" / f"tutorial_mode_{MODE_TO_INSPECT}"

if not result_dir.exists():
    print(
        f"No tutorial results found for Mode {MODE_TO_INSPECT}. "
        f"Add {MODE_TO_INSPECT} to RUN_MODES and execute its section first."
    )
else:
    produced = sorted(path.relative_to(result_dir).as_posix() for path in result_dir.rglob("*") if path.is_file())
    display(pd.DataFrame({"Produced file": produced}))

    predictions_path = result_dir / "Table" / "Predictions.csv"
    if predictions_path.is_file():
        display(pd.read_csv(predictions_path).head())

    log_path = result_dir / "process.log"
    if log_path.is_file():
        print("\nLast 15 log lines:\n")
        print("\n".join(log_path.read_text().splitlines()[-15:]))

No tutorial results found for Mode 4. Add 4 to RUN_MODES and execute its section first.


## Recommended order

1. Run the installation and data-inspection sections.
2. Run Mode 4 to verify the environment with the toy dataset.
3. Run Mode 3 to see DAP training plus labeled inference.
4. Run Modes 1 or 2 for unlabeled samples.
5. Run Mode 6 for the fixed paper reproduction.
6. Run Mode 5 only when full-panel retraining is required and sufficient memory
   is available.

For a clean repeat, use a new `result_folder_path`. Reusing a result directory
can intentionally reuse model and intermediate files from an earlier run.